In [1]:
# Notebook 03: Full dataset profile — row counts, class sizes, imbalance, feature ranges.
# Output: a per-class summary table saved to results/, used to design the train/test/val split.

import os
import time
from pathlib import Path
import pandas as pd
import numpy as np

from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/composite_resilience_framework')
DATA_ROOT = Path('/content/scratch/CICIoT2023_CSV/CSV')
RESULTS = PROJECT / 'results'

# If we lost the extraction (new Colab session), re-extract from Drive
if not DATA_ROOT.exists():
    print("Extracted dataset not found locally — re-extracting from Drive...")
    import zipfile, shutil
    zip_drive = PROJECT / 'data' / 'raw' / 'CSV.zip'
    zip_local = Path('/content/scratch/CSV.zip')
    Path('/content/scratch').mkdir(exist_ok=True)
    shutil.copy(zip_drive, zip_local)
    with zipfile.ZipFile(zip_local) as z:
        z.extractall('/content/scratch/CICIoT2023_CSV')
    print("Re-extraction complete.\n")

assert DATA_ROOT.exists(), f"DATA_ROOT missing: {DATA_ROOT}"

# --- Step 1: per-file row counts (fast, no full load) ---
print("=== Step 1: Per-class profile (row counts + size) ===")
print("This iterates all 309 files; expect ~30-60 seconds.\n")

class_folders = sorted([p for p in DATA_ROOT.iterdir() if p.is_dir()])
records = []
t0 = time.time()
for cf in class_folders:
    csvs = sorted(cf.glob('*.csv'))
    cls_rows = 0
    cls_bytes = 0
    for c in csvs:
        with open(c) as fp:
            n = sum(1 for _ in fp) - 1
        cls_rows += n
        cls_bytes += c.stat().st_size
    records.append({
        'class': cf.name,
        'files': len(csvs),
        'rows': cls_rows,
        'size_MB': cls_bytes / 1024 / 1024,
    })
df_profile = pd.DataFrame(records).sort_values('rows', ascending=False).reset_index(drop=True)
print(f"  Profiled in {time.time()-t0:.1f}s\n")

# Category mapping (matches CIC's 7-category taxonomy + Benign)
def categorize(cls):
    if cls.startswith('DDoS-'): return 'DDoS'
    if cls.startswith('DoS-'): return 'DoS'
    if cls.startswith('Mirai-'): return 'Mirai'
    if cls.startswith('Recon-'): return 'Recon'
    if cls == 'Benign_Final': return 'Benign'
    if cls == 'DictionaryBruteForce': return 'BruteForce'
    if cls in ('DNS_Spoofing', 'MITM-ArpSpoofing'): return 'Spoofing'
    if cls in ('Backdoor_Malware','BrowserHijacking','CommandInjection',
               'SqlInjection','Uploading_Attack','XSS'): return 'Web'
    return 'Other'
df_profile['category'] = df_profile['class'].map(categorize)

# Per-class summary
print("=== Per-class summary (sorted by row count, descending) ===")
print(df_profile.to_string(index=False, formatters={
    'rows': '{:,}'.format, 'size_MB': '{:,.1f}'.format
}))

# Per-category summary
print("\n=== Per-category summary ===")
cat_summary = (df_profile.groupby('category')
               .agg(classes=('class','count'),
                    files=('files','sum'),
                    rows=('rows','sum'),
                    size_MB=('size_MB','sum'))
               .sort_values('rows', ascending=False))
total_rows = cat_summary['rows'].sum()
cat_summary['pct_of_total'] = 100 * cat_summary['rows'] / total_rows
print(cat_summary.to_string(formatters={
    'rows': '{:,}'.format, 'size_MB': '{:,.1f}'.format,
    'pct_of_total': '{:5.2f}%'.format
}))
print(f"\n  GRAND TOTAL: {total_rows:,} rows")

# Imbalance ratios
print("\n=== Imbalance picture ===")
biggest = df_profile.iloc[0]
smallest = df_profile.iloc[-1]
print(f"  Largest class : {biggest['class']:<25} {biggest['rows']:>15,} rows")
print(f"  Smallest class: {smallest['class']:<25} {smallest['rows']:>15,} rows")
print(f"  Ratio (largest:smallest) = {biggest['rows']/max(smallest['rows'],1):,.0f} : 1")

# --- Step 2: feature-range sanity check on a random sample ---
print("\n=== Step 2: Feature-range sanity check (per-class, 1000 random rows) ===")
print("Checking for NaNs, infs, and value ranges across all classes...\n")
nan_total = 0
inf_total = 0
issues = []
for cls in df_profile['class']:
    f = next((DATA_ROOT / cls).glob('*.csv'))
    df_s = pd.read_csv(f, nrows=1000)
    n_nan = df_s.isna().sum().sum()
    # infs only meaningful in numeric cols
    num = df_s.select_dtypes(include=[np.number])
    n_inf = np.isinf(num.values).sum() if len(num) > 0 else 0
    nan_total += n_nan
    inf_total += n_inf
    if n_nan or n_inf:
        issues.append((cls, n_nan, n_inf))
print(f"  Total NaNs across class samples: {nan_total}")
print(f"  Total infs across class samples: {inf_total}")
if issues:
    print(f"  Classes with issues:")
    for c, n, i in issues:
        print(f"    {c}: NaNs={n}, infs={i}")
else:
    print("  All sampled classes clean (no NaNs, no infs).")

# --- Step 3: save profile to Drive ---
RESULTS.mkdir(exist_ok=True)
out_csv = RESULTS / 'dataset_profile.csv'
df_profile.to_csv(out_csv, index=False)
out_cat = RESULTS / 'dataset_profile_by_category.csv'
cat_summary.to_csv(out_cat)
print(f"\n=== Saved ===")
print(f"  {out_csv}")
print(f"  {out_cat}")
print("\nReady for split design.")

Mounted at /content/drive
Extracted dataset not found locally — re-extracting from Drive...
Re-extraction complete.

=== Step 1: Per-class profile (row counts + size) ===
This iterates all 309 files; expect ~30-60 seconds.

  Profiled in 7.6s

=== Per-class summary (sorted by row count, descending) ===
                  class  files      rows size_MB   category
        DDoS-ICMP_Flood     27 7,200,501 1,264.8       DDoS
         DDoS-UDP_Flood     21 5,412,231   977.5       DDoS
         DDoS-TCP_Flood     18 4,497,649   804.7       DDoS
      DDoS-PSHACK_FLOOD     16 4,094,772   733.9       DDoS
         DDoS-SYN_Flood     16 4,059,179   738.4       DDoS
       DDoS-RSTFINFLOOD     16 4,045,279   731.5       DDoS
DDoS-SynonymousIP_Flood     14 3,598,133   643.4       DDoS
          DoS-UDP_Flood     17 3,072,993   577.4        DoS
          DoS-TCP_Flood     11 2,671,430   481.9        DoS
          DoS-SYN_Flood      8 2,028,836   372.5        DoS
           Benign_Final      4 1,098